In [9]:
import pandas as pd
import fasttext
import os

for dirname,_,filenames in os.walk("D:/fastText_trained_model_and_datas/ftt_input"):
    for filename in filenames:
        print(os.path.join(dirname,filename))


D:/fastText_trained_model_and_datas/ftt_input\DBPEDIA_test.csv
D:/fastText_trained_model_and_datas/ftt_input\DBPEDIA_train.csv
D:/fastText_trained_model_and_datas/ftt_input\DBPEDIA_val.csv
D:/fastText_trained_model_and_datas/ftt_input\DBP_wiki_data.csv


In [10]:
train_file = "D:/fastText_trained_model_and_datas/ftt_input/DBPEDIA_train.csv"
df = pd.read_csv(train_file)
df.head()

,text,l1,l2,l3
0,"William Alexander Massey (October 7, 1856 – Ma...",Agent,Politician,Senator
1,Lions is the sixth studio album by American ro...,Work,MusicalWork,Album
2,"Pirqa (Aymara and Quechua for wall, hispaniciz...",Place,NaturalPlace,Mountain
3,Cancer Prevention Research is a biweekly peer-...,Work,PeriodicalLiterature,AcademicJournal
4,The Princeton University Chapel is located on ...,Place,Building,HistoricBuilding


In [11]:
# 把dataframe的列名称修改一先，使得它更加有意义
df.rename(columns={"text":"Texts","l1":"class",'l2':'profession','l3':"Type"},inplace=True)
df.head()

,Texts,class,profession,Type
0,"William Alexander Massey (October 7, 1856 – Ma...",Agent,Politician,Senator
1,Lions is the sixth studio album by American ro...,Work,MusicalWork,Album
2,"Pirqa (Aymara and Quechua for wall, hispaniciz...",Place,NaturalPlace,Mountain
3,Cancer Prevention Research is a biweekly peer-...,Work,PeriodicalLiterature,AcademicJournal
4,The Princeton University Chapel is located on ...,Place,Building,HistoricBuilding


In [12]:
test_file = "D:/fastText_trained_model_and_datas/ftt_input/DBPEDIA_test.csv"
df_test = pd.read_csv(test_file)
df_test.rename(columns = {'text':'Texts','l1':'class', 'l2':'profession','l3':'Type'}, inplace = True)
df_test.head()

,Texts,class,profession,Type
0,Liu Chao-shiuan (Chinese: 劉兆玄; pinyin: Liú Zhà...,Agent,Politician,PrimeMinister
1,"Michelle Maylene (born January 20, 1987) is an...",Agent,Actor,AdultActor
2,Hirfanlı Dam is a dam in Turkey. The developme...,Place,Infrastructure,Dam
3,Grote Prijs Stad Zottegem is a single-day road...,Event,Race,CyclingRace
4,"Johannes Petrus \""Hans\"" Nijman (September 23,...",Agent,Athlete,MartialArtist


In [13]:
print(f"Train:{df.shape},Test:{df_test.shape}")

Train:(240942, 4),Test:(60794, 4)


In [14]:
df['class'].value_counts()

class
Agent             124798
Place              45877
Species            21472
Work               21013
Event              19106
SportsSeason        5883
UnitOfWork          1761
TopicalConcept       784
Device               248
Name: count, dtype: int64

In [15]:
df['profession'].unique()

<ArrowStringArray>
[                  'Politician',                  'MusicalWork',
                 'NaturalPlace',         'PeriodicalLiterature',
                     'Building',                       'Animal',
                 'Organisation',                       'Person',
                      'Athlete',                   'Settlement',
                    'LegalCase',              'MotorcycleRider',
                      'Company',        'RouteOfTransportation',
                'SocietalEvent',            'WinterSportPlayer',
 'ClericalAdministrativeRegion',       'EducationalInstitution',
                  'BodyOfWater',                        'Plant',
               'Infrastructure',         'FootballLeagueSeason',
                        'Actor',                'SportsManager',
                       'Cleric',                        'Boxer',
                      'Cartoon',                        'Venue',
                       'Artist',                   'Tournament',
      

In [16]:
df['Type'].value_counts()

Type
AcademicJournal          1924
Manga                    1924
FigureSkater             1924
OlympicEvent             1923
Gymnast                  1922
                         ... 
Cycad                     145
AnimangaCharacter         144
BeachVolleyballPlayer     137
CanadianFootballTeam      133
BiologicalDatabase        129
Name: count, Length: 219, dtype: int64

In [17]:
professions = {}
i = 0
for Name in df['profession'].unique():
    professions[Name] = i
    i += 1
df['Class'] = df['profession'].map(professions)    
df_test['Class'] = df_test['profession'].map(professions)    

df.drop("class",axis=1,inplace=True)
df_test.drop("class",axis=1,inplace=True)
df.head()


,Texts,profession,Type,Class
0,"William Alexander Massey (October 7, 1856 – Ma...",Politician,Senator,0
1,Lions is the sixth studio album by American ro...,MusicalWork,Album,1
2,"Pirqa (Aymara and Quechua for wall, hispaniciz...",NaturalPlace,Mountain,2
3,Cancer Prevention Research is a biweekly peer-...,PeriodicalLiterature,AcademicJournal,3
4,The Princeton University Chapel is located on ...,Building,HistoricBuilding,4


In [18]:
# Lets do some cleaning of this text
def clean_it(text,normalize=True):
    # Replacing possible issues with data. We can add or reduce the replacemtent in this chain
    s = str(text).replace(',',' ').replace('"','').replace('\' ',' \' ').replace('.',' . ').replace('(',' ( ').\
            replace(')',' ) ').replace('!',' ! ').replace('?',' ? ').replace(':',' ').replace(';',' ').lower()
    
    # normalizing / encoding the text
    if normalize:
        s = s.normalize('NFKD').str.encode('ascii','ignore').str.decode('utf-8')
    
    return s

In [19]:
'__class__' + df['Class'].iloc[0].astype(str) + ' '

'__class__0 '

In [20]:
# Now lets define a small function where we can use above cleaning on datasets
def clean_df(data, cleanit= False, shuffleit=False, encodeit=False, label_prefix='__class__'):
    # Defining the new data
    df = data[['Type','Texts']].copy(deep=True)
    df['Class'] = label_prefix + data['Class'].astype(str) + ' '
    
    # cleaning it
    if cleanit:
        df['Type'] = df['Type'].apply(lambda x: clean_it(x,encodeit))
        df['Texts'] = df['Texts'].apply(lambda x: clean_it(x,encodeit))
    
    # shuffling it
    if shuffleit:
        df.sample(frac=1).reset_index(drop=True)
            
    return df

In [22]:
%%time
# Transform the datasets using the above clean functions
df_train_cleaned = clean_df(df,True,True)
df_test_cleaned = clean_df(df_test,True,True)

CPU times: total: 2.86 s
Wall time: 3.13 s


In [23]:
df_train_cleaned.head()

,Type,Texts,Class
0,senator,william alexander massey ( october 7 1856 – ...,__class__0
1,album,lions is the sixth studio album by american ro...,__class__1
2,mountain,pirqa ( aymara and quechua for wall hispanic...,__class__2
3,academicjournal,cancer prevention research is a biweekly peer-...,__class__3
4,historicbuilding,the princeton university chapel is located on ...,__class__4


Write files to disk as fastText classifier API reads files from disk.

In [25]:
# Write files to disk as fastText classifier API reads files from disk.
from email import header


train_file = "D:/fastText_trained_model_and_datas/cleaneddata/dbpedia_train.csv"
df_train_cleaned.to_csv(train_file,header=None,index=False,columns=['Class','Type','Texts'])

test_file = "D:/fastText_trained_model_and_datas/cleaneddata/dbpedia_test.csv"
df_test_cleaned.to_csv(test_file,header=None,index=False,columns=['Class','Type','Texts'])

Fast Text processing

In [27]:
%%time
## Using fastText for feature extraction and training
from fasttext import train_supervised 
"""
fastText expects and training file (csv), a model name as input arguments.
label_prefix refers to the prefix before label string in the dataset.
default is __label__. In our dataset, it is __class__. 
There are several other parameters which can be seen in: 
https://pypi.org/project/fasttext/
"""

model = train_supervised(input=train_file,label='__class__',lr=1.0,epoch=75,loss='ova',wordNgrams=2,dim=200,thread=2,verbose=100)

CPU times: total: 49min 59s
Wall time: 26min 22s


In [28]:
# Save the model
model.save_model("D:/fastText_trained_model_and_datas/ftt_model/ftt_model.bin")

In [29]:
# Load the model
fastModel = fasttext.load_model("D:/fastText_trained_model_and_datas/ftt_model/ftt_model.bin")
fastModel

In [30]:
print('Number of words :',len(fastModel.words))
print('Label :',len(fastModel.labels))

Number of words : 753758
Label : 70


Evaluation

In [31]:
for k in range(1,6):
    results = fastModel.test(test_file,k=k)
    print(f"Test Samples: {results[0]} Precision@{k} : {results[1]*100:2.4f} Recall@{k} : {results[2]*100:2.4f}")

Test Samples: 60794 Precision@1 : 96.6526 Recall@1 : 96.6526
Test Samples: 60794 Precision@2 : 49.1899 Recall@2 : 98.3798
Test Samples: 60794 Precision@3 : 32.8530 Recall@3 : 98.5591
Test Samples: 60794 Precision@4 : 24.6480 Recall@4 : 98.5920
Test Samples: 60794 Precision@5 : 19.8098 Recall@5 : 99.0492


In [34]:
# Predict
print('Text :',df_test_cleaned['Texts'].iloc[0])
print('Actual :',df_test_cleaned['Class'].iloc[0])
text = df_test_cleaned['Texts'].iloc[0]
print('Prediction :',fastModel.predict([text]))

Text : liu chao-shiuan  ( chinese  劉兆玄  pinyin  liú zhàoxuán  born may 10  1943 )  is a taiwanese educator and politician .  he is a former president of the national tsing hua university  ( 1987–1993 )  and soochow university  ( 2004–2008 )  and a former premier of the republic of china  ( 2008–2009 )  . 
Actual : __class__0 
Prediction : ([['__class__0']], [array([1.00001], dtype=float32)])
